<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L08-drift-and-the-alarm-audit/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L08-drift-and-the-alarm-audit/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/predictive-maintenance/lessons/P03-L08-drift-and-the-alarm-audit/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/predictive-maintenance/lessons/P03-L08-drift-and-the-alarm-audit/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P03-L08 · Drift, retraining, and the alarm audit

**You will build:** the half of a monitoring programme that nobody budgets for — the part
that runs after go-live. An unsupervised drift detector on the feature distribution, a
diagnosis that tells three kinds of drift apart, the three *different* fixes they need, and
the alarm audit that feeds the plant's own invoices back into module 1's cost function so
the threshold stops being a number somebody guessed at kick-off.

**Time:** ~80 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (the profiler and the tier gate),
`P03-L01-alarm-economics` (the threshold sweep, the lead-time rule and the three prices,
all carried forward here), `P03-L07-ot-network-deployment` (the one-way path: the model
ships data out and gets nothing automatic back, which is why the detector below never sees
a label).

By the end you will be able to:

1. Implement population stability index over reference-derived bins, and measure what "no
   drift" looks like on a plant by scoring the reference window against itself.
2. Implement a regime-conditional drift statistic and explain why a drift that vanishes
   once you condition on the duty point is not a drift in the machine.
3. Diagnose which of three drifts you have — instrument, regime, ageing — from three
   unsupervised statistics, in the order that makes a moved instrument outrank both others.
4. Implement the three fixes (re-baseline, condition, re-price) and measure the cost of
   applying the right fix to the wrong drift.
5. Estimate the three prices from an alarm audit, name the ones still resting on a guess,
   and re-derive the threshold from the plant's own history rather than from kick-off.

**The data is synthetic and the generator is in this notebook.** Nothing is downloaded.
To grade a *diagnosis* you have to know which drift was injected, and no plant historian
ships with that column. `meta.yaml` declares the generator under `datasets:`. The method
transfers; the particular numbers describe no real plant.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import sys
import time
import traceback
from typing import Callable, NamedTuple, Sequence

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The fleet. 300 units, one health-index reading per hour for 20 days, in five parallel
# worlds: a reference window and four things that can happen to it afterwards. Module 3 built
# the health index; here it is a column of numbers that reads about 1.0 on a well unit.
N_UNITS = 300
N_HOURS = 480
SEED = 20260922

# The duty point. Every reading carries the regime it was taken in, because the process
# historian knows the duty point — it is a setpoint, not something you have to infer.
N_REGIMES = 3
REGIME_NAMES = ("low", "normal", "high")
REGIME_EFFECT = (0.00, 0.00, 0.55)    # what duty alone adds to a HEALTHY unit's index
REGIME_BLOCK = 8                      # hours the duty point holds before it is redrawn

# The instrument check. Every transmitter runs a self-test against its own internal
# reference on a fixed schedule and reports a number that has nothing to do with the machine.
# This is the check standard the drift diagnosis leans on, and section 6 is about why you
# cannot tell an instrument from a machine without one.
N_CHECKS = 48
CHECK_TRUE = 1.000

# Module 1's maintenance contract, carried forward unchanged: an alarm that arrives with less
# warning than this prevented nothing.
LEAD_HOURS = 24

# Drift-detector settings. N_BINS is the resolution of the histogram the PSI is computed on;
# PSI_FLOOR keeps an empty bin from taking a logarithm of zero. The ALERT LEVEL is not here,
# because it is not a constant: section 3 measures it on this plant.
N_BINS = 12
PSI_FLOOR = 1e-6
NULL_DRAWS = 25
EXPLAINED_FRACTION = 0.5    # how much of the marginal drift the regime must explain
QUIET_Q = 0.10              # the quantile a re-baseline calls a unit's "quiet" level

THRESHOLDS = np.round(np.arange(1.00, 4.51, 0.05), 2)

# The three prices somebody wrote on a whiteboard at kick-off. Section 8 replaces two of them
# with the plant's own invoices and prints which one is still a guess.
MIN_OBSERVATIONS = 5

# The four things you can do about drift, and the four things it can be.
FIXES = ("none", "rebaseline", "condition", "reprice")
DIAGNOSES = ("none", "recalibration", "regime", "ageing")

# What an inspection finds, in the audit log's own vocabulary.
OUTCOMES = ("planned", "unplanned", "false_alarm")

# A run ends one of two ways in this module: it failed, or the window closed on it.
FAILURE = "failure"
CENSORED = "censored"


class Run(NamedTuple):
    """One unit's observation window, carried forward from module 4."""
    unit: int
    end_hour: int
    kind: str          # FAILURE or CENSORED


class Prices(NamedTuple):
    """Module 1's cost model. Three numbers, and the threshold is a function of them."""
    planned: float
    unplanned: float
    false_alarm: float


class Counts(NamedTuple):
    """The confusion matrix, in module 4's field order."""
    tp: int
    fn: int
    fp: int
    tn: int


class OperatingPoint(NamedTuple):
    threshold: float
    cost: float
    counts: Counts


class Window(NamedTuple):
    """One observation window of the whole fleet, and the truth about it."""
    name: str
    health: np.ndarray     # (n_units, n_hours) the health index the model sees
    regime: np.ndarray     # (n_units, n_hours) duty point, 0..N_REGIMES-1, from the historian
    check: np.ndarray      # (n_units, N_CHECKS) instrument self-test readings
    runs: tuple            # tuple[Run, ...], one per unit


class Baseline(NamedTuple):
    """What the kick-off deployment measured and froze. Everything later is compared to this."""
    quiet_level: float        # the reference window's QUIET_Q quantile, pooled
    regime_offset: tuple      # per-regime median minus the overall median, on healthy data
    check_level: float        # the reference window's median instrument check reading
    threshold: float          # the cost-optimal threshold at the kick-off prices
    prices: Prices            # the kick-off guess


class DriftSignature(NamedTuple):
    """Three unsupervised numbers and one descriptive one. No labels anywhere."""
    marginal: float       # PSI of the feature, ignoring duty point
    conditional: float    # PSI of the feature within duty regime
    check: float          # PSI of the instrument self-test readings
    median_shift: float   # current median minus reference median, for reporting only


class AuditEntry(NamedTuple):
    """One line of the alarm audit: an alarm raised, or a failure that arrived without one."""
    unit: int
    hour: int
    outcome: str      # one of OUTCOMES
    cost: float       # what the job actually cost, off the invoice


class PriceEstimate(NamedTuple):
    prices: Prices
    counts: tuple            # (n_planned, n_unplanned, n_false_alarm)
    still_guessed: tuple     # the OUTCOMES that fell back to the kick-off number


class FixResult(NamedTuple):
    fix: str
    threshold: float
    counts: Counts
    cost: float


class Response(NamedTuple):
    diagnosis: str
    fix: str
    result: FixResult


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("reference_bins",),
    "exercise 2": ("psi",),
    "exercise 3": ("conditional_psi",),
    "exercise 4": ("drift_signature",),
    "exercise 5": ("diagnose_drift",),
    "exercise 6": ("adjust_feature",),
    "exercise 7": ("estimate_prices",),
    "exercise 8": ("apply_fix",),
    "exercise 9": ("respond_to_drift",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (conditional_psi)"; several -> "exercises 1, 2 and 7"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)

## 1. Five windows, and the machinery you already own

Below is the generator. Read it — knowing exactly what was injected is the only reason a
*diagnosis* can be graded at all. It builds five observation windows of the same fleet:

- **reference** — the window the model was fitted and priced on at kick-off.
- **steady** — a later window in which nothing happened. The detector has to stay quiet.
- **recalibration** — a contractor swapped transmitters on part of the fleet during the
  turnaround. Those channels now read high by a per-unit amount, and their **self-test
  readings moved with them**. The machines are fine.
- **regime** — a parallel train came off line, so the surviving units run at high duty.
  The feature rises because duty rises. The machines are fine.
- **ageing** — three years of running hours. Every unit is genuinely a little worse, more
  of them are on their way to failure, and the instruments are perfect.

Two of those raise the feature and need opposite responses. Everything after this is about
telling them apart without a single label, because module 7 left you no way back from the
plant.

In [ ]:
def _make_window(rng: np.random.Generator, name: str, n_units: int, n_hours: int,
                 fail_fraction: float, regime_mix: Sequence[float],
                 calib_fraction: float = 0.0, calib_range: tuple = (0.0, 0.0),
                 pf_range: tuple = (0.15, 0.35), ramp_exponent: float = 1.7,
                 severity_range: tuple = (0.45, 3.0)) -> Window:
    """Build one observation window. Given to you; the docstring is the specification.

    Four independent knobs, one per thing that can go wrong:
      * `fail_fraction` — how much of the fleet runs to failure inside the window. Ageing
        raises it; a base-rate change is exactly this and nothing else.
      * `regime_mix` — the duty-point mix the plant is running (a regime change moves it).
        Duty is drawn per shift of `REGIME_BLOCK` hours, per unit, so EVERY unit sees every
        regime. That matters in section 7: a unit's own quiet hours are its low-duty hours,
        which is why re-baselining against them does not remove a duty change.
      * `calib_fraction` / `calib_range` — a per-unit additive error on the instrument. It
        lands on the health index AND on the instrument's self-test reading, because a
        transmitter that reads high reads high about everything.
      * `pf_range` / `ramp_exponent` / `severity_range` — how long before the end of a run
        the defect becomes visible, how sharply, and how loudly. An old fleet's defects have been developing since before
        the window opened, so its P-F intervals are long and its ramps are gradual: a failing
        unit's own QUIET hours already read high. Nothing here touches the self-test, because
        the machine is worse and the instrument is fine. That line and the calibration line
        above are the whole diagnostic problem.

    After a unit fails it is repaired, so the hours from `end_hour` on read like a new
    machine. Nothing scores them — every sweep here slices `[:end_hour]`, as module 4's did.
    """
    blocks = int(np.ceil(n_hours / REGIME_BLOCK))
    drawn = rng.choice(N_REGIMES, size=(n_units, blocks), p=list(regime_mix))
    regime = np.repeat(drawn, REGIME_BLOCK, axis=1)[:, :n_hours]

    unit_level = rng.normal(1.00, 0.02, size=n_units)
    effect = np.asarray(REGIME_EFFECT, dtype=float)
    quiet = unit_level[:, None] + effect[regime]
    health = quiet + rng.normal(0.0, 0.055, size=(n_units, n_hours))
    shock = (rng.random((n_units, n_hours)) < 0.002) * rng.uniform(0.25, 0.70, (n_units, n_hours))
    health = health + shock

    hours = np.arange(n_hours)
    end_hour = np.full(n_units, n_hours, dtype=int)
    kinds = [CENSORED] * n_units
    n_fail = int(round(fail_fraction * n_units))
    for unit in rng.choice(n_units, size=n_fail, replace=False):
        end = int(rng.integers(int(0.45 * n_hours), n_hours + 1))
        p_f = int(rng.integers(int(pf_range[0] * n_hours), int(pf_range[1] * n_hours) + 1))
        severity = float(rng.uniform(severity_range[0], severity_range[1]))
        ramp = np.clip((hours - (end - p_f)) / p_f, 0.0, 1.0) ** float(ramp_exponent)
        ramp[hours >= end] = 0.0
        health[unit] = health[unit] + severity * ramp
        if end < n_hours:
            health[unit, end:] = (quiet[unit, end:]
                                  + rng.normal(0.0, 0.055, size=n_hours - end))
        end_hour[unit], kinds[unit] = end, FAILURE

    calib = np.zeros(n_units)
    n_calib = int(round(calib_fraction * n_units))
    if n_calib:
        who = rng.choice(n_units, size=n_calib, replace=False)
        calib[who] = rng.uniform(calib_range[0], calib_range[1], size=n_calib)
    health = health + calib[:, None]
    check = CHECK_TRUE + rng.normal(0.0, 0.02, (n_units, N_CHECKS)) + calib[:, None]

    runs = tuple(Run(int(u), int(end_hour[u]), kinds[u]) for u in range(n_units))
    return Window(name, health, regime, check, runs)


BASE_MIX = (0.35, 0.55, 0.10)
HIGH_MIX = (0.02, 0.18, 0.80)
BASE_FAIL = 0.10
AGED_FAIL = 0.40
AGED_PF = (0.80, 1.00)
AGED_RAMP = 0.7
AGED_SEVERITY = (0.25, 1.20)


def generate_fleet(seed: int = SEED, n_units: int = N_UNITS, n_hours: int = N_HOURS
                   ) -> dict:
    """Deterministic synthetic fleet: five windows of the same plant. Given to you.

    Returns an ordered mapping name -> Window. Every window is a fresh draw of the same
    healthy population; only the named knob differs. The last one, `recal_and_regime`, is the
    turnaround that did both at once, and it exists to grade the ORDER of the diagnosis.
    """
    rng = np.random.default_rng(seed)
    spec = (
        ("reference", dict(fail_fraction=BASE_FAIL, regime_mix=BASE_MIX)),
        ("steady", dict(fail_fraction=BASE_FAIL, regime_mix=BASE_MIX)),
        ("recalibration", dict(fail_fraction=BASE_FAIL, regime_mix=BASE_MIX,
                               calib_fraction=0.45, calib_range=(0.45, 1.15))),
        ("regime", dict(fail_fraction=BASE_FAIL, regime_mix=HIGH_MIX)),
        ("ageing", dict(fail_fraction=AGED_FAIL, regime_mix=BASE_MIX,
                        pf_range=AGED_PF, ramp_exponent=AGED_RAMP,
                        severity_range=AGED_SEVERITY)),
        ("recal_and_regime", dict(fail_fraction=BASE_FAIL, regime_mix=HIGH_MIX,
                                  calib_fraction=0.45, calib_range=(0.45, 1.15))),
    )
    return {name: _make_window(rng, name, n_units, n_hours, **kw) for name, kw in spec}


def halve(window: Window) -> tuple:
    """Split a window into (history, live) by unit, renumbering each half from zero.

    Given to you. The history half is what the audit has already seen; the live half is the
    fleet you are about to protect. Re-deriving a threshold on the half you then score it on
    would flatter every fix equally and hide the differences this lesson is about.
    """
    n = window.health.shape[0]
    cut = n // 2
    out = []
    for lo, hi in ((0, cut), (cut, n)):
        runs = tuple(Run(i, window.runs[u].end_hour, window.runs[u].kind)
                     for i, u in enumerate(range(lo, hi)))
        out.append(Window(f"{window.name}[{lo}:{hi}]", window.health[lo:hi],
                          window.regime[lo:hi], window.check[lo:hi], runs))
    return out[0], out[1]

Three functions from earlier modules, given to you so this notebook grades the new material
and not module 1 again: the sweep, the confusion matrix with its lead-time rule, and the
price of that matrix. `alarm_counts` is module 4's rule exactly — the alarm window stops at
`end_hour`, and a failure caught with less than `LEAD_HOURS` of warning is a false negative.

In [ ]:
def sweep_counts(health: np.ndarray, runs: Sequence[Run], thresholds: np.ndarray,
                 lead_hours: int) -> tuple:
    """Confusion-matrix counts at every threshold at once. Given to you.

    The running maximum of a unit's health is non-decreasing, so the first hour it crosses a
    threshold is one `searchsorted` away — the whole sweep is O(hours + thresholds) per unit
    instead of one pass per threshold.
    """
    thr = np.asarray(thresholds, dtype=float)
    k = thr.size
    tp = np.zeros(k, dtype=int); fn = np.zeros(k, dtype=int)
    fp = np.zeros(k, dtype=int); tn = np.zeros(k, dtype=int)
    for run in runs:
        row = np.asarray(health, dtype=float)[run.unit, :run.end_hour]
        if row.size == 0:
            idx = np.zeros(k, dtype=int)
            alarmed = np.zeros(k, dtype=bool)
        else:
            idx = np.searchsorted(np.maximum.accumulate(row), thr, side="left")
            alarmed = idx < row.size
        if run.kind == FAILURE:
            caught = alarmed & ((run.end_hour - idx) >= lead_hours)
            tp += caught; fn += ~caught
        else:
            fp += alarmed; tn += ~alarmed
    return tp, fn, fp, tn


def alarm_counts(health: np.ndarray, runs: Sequence[Run], threshold: float,
                 lead_hours: int = LEAD_HOURS) -> Counts:
    """Module 4's confusion matrix at one threshold. Given to you."""
    tp, fn, fp, tn = sweep_counts(health, runs, np.array([float(threshold)]), lead_hours)
    return Counts(int(tp[0]), int(fn[0]), int(fp[0]), int(tn[0]))


def expected_cost(counts: Counts, prices: Prices) -> float:
    """Module 1's cost function. True negatives are free. Given to you."""
    return float(counts.tp * prices.planned + counts.fn * prices.unplanned
                 + counts.fp * prices.false_alarm)


def cost_optimal_threshold(health: np.ndarray, runs: Sequence[Run], thresholds: np.ndarray,
                           prices: Prices, lead_hours: int = LEAD_HOURS) -> OperatingPoint:
    """Module 1's sweep, lowest tied threshold wins. Given to you."""
    thr = np.asarray(thresholds, dtype=float)
    if thr.size == 0:
        raise ValueError("thresholds is empty: there is nothing to sweep")
    tp, fn, fp, tn = sweep_counts(health, runs, thr, lead_hours)
    cost = tp * prices.planned + fn * prices.unplanned + fp * prices.false_alarm
    best = int(np.argmin(cost))
    return OperatingPoint(float(thr[best]), float(cost[best]),
                          Counts(int(tp[best]), int(fn[best]), int(fp[best]), int(tn[best])))


def fit_baseline(reference: Window, prices: Prices, thresholds: np.ndarray = THRESHOLDS,
                 lead_hours: int = LEAD_HOURS, quiet_q: float = QUIET_Q) -> Baseline:
    """Everything the kick-off deployment measured and froze. Given to you."""
    overall = float(np.median(reference.health))
    offsets = tuple(float(np.median(reference.health[reference.regime == r]) - overall)
                    for r in range(N_REGIMES))
    point = cost_optimal_threshold(reference.health, reference.runs, thresholds, prices,
                                   lead_hours)
    return Baseline(quiet_level=float(np.quantile(reference.health, quiet_q)),
                    regime_offset=offsets,
                    check_level=float(np.median(reference.check)),
                    threshold=float(point.threshold), prices=prices)


KICKOFF = Prices(planned=9_000.0, unplanned=30_000.0, false_alarm=15_000.0)
FLEET = generate_fleet()
REFERENCE = FLEET["reference"]
BASELINE = fit_baseline(REFERENCE, KICKOFF)

# Every window is split once, here, and used the same way everywhere below: the drift is
# DIAGNOSED against the reference window, the threshold is RE-DERIVED on the history half,
# and every fix is SCORED on the live half it has never seen.
HISTORY, LIVE = {}, {}
for _name, _win in FLEET.items():
    HISTORY[_name], LIVE[_name] = halve(_win)

Run this. It is the plant as the kick-off deployment left it: one threshold, three prices,
and a base rate measured on the reference window.

In [ ]:
def _show_the_plant() -> None:
    print(f"fleet: {N_UNITS} units x {N_HOURS} h, {len(FLEET)} windows, "
          f"{N_CHECKS} instrument self-tests per unit per window")
    print(f"kick-off prices: planned {KICKOFF.planned:,.0f} · unplanned "
          f"{KICKOFF.unplanned:,.0f} · false alarm {KICKOFF.false_alarm:,.0f}")
    print(f"kick-off threshold (cost-optimal on the reference window): "
          f"{BASELINE.threshold:.2f}")
    print(f"kick-off quiet level (q={QUIET_Q:.2f}): {BASELINE.quiet_level:.3f} · "
          f"instrument check level: {BASELINE.check_level:.3f}")
    print("  regime offsets measured on healthy reference data: "
          + " · ".join(f"{REGIME_NAMES[r]} {BASELINE.regime_offset[r]:+.3f}"
                       for r in range(N_REGIMES)))
    print()
    print(f"  {'window':<18}{'failures':>9}{'base rate':>11}{'median h':>10}"
          f"{'median check':>14}")
    for name, win in FLEET.items():
        fails = sum(1 for r in win.runs if r.kind == FAILURE)
        print(f"  {name:<18}{fails:>9}{fails / len(win.runs):>11.3f}"
              f"{float(np.median(win.health)):>10.3f}{float(np.median(win.check)):>14.3f}")
    print("\nthree of those windows raise the median health index. One of the three is the")
    print("only one where the machines actually got worse, and the median cannot tell you")
    print("which. That is the whole of sections 3 to 6.")


_try("the plant", _show_the_plant)

## 2. Exercise 1 — `reference_bins()`

Everything that follows compares a histogram of the reference window with a histogram of a
later one. The bins have to come from the **reference alone**. Pool the two windows to
choose bins and the bins move with the drift, which is the neatest way to build a detector
that reports nothing has happened.

Use quantiles of the reference, so each bin starts with the same share of the mass, and put
infinities on the outside so no later reading can fall off the end. A reference with heavy
ties can produce duplicate cut points; collapse them rather than dividing by a zero-width
bin.

<details><summary>💡 Hint 1 — what to think about</summary>

Picture a reference where four readings in five sit on one low value and the fifth is far
out. A cut that shares out the *readings* lands in one place; a cut that shares out the
*range* lands somewhere nothing lives. Which one does a PSI need? And where does a later
reading go if it is lower than anything the reference ever produced?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Flatten and validate first: empty input, any non-finite value, fewer than two bins. Take the
reference's quantiles at evenly spaced probabilities strictly between 0 and 1, one fewer of
them than there are bins. Keep each distinct cut point once, in order, then put minus
infinity in front and plus infinity at the end. The current window plays no part.
</details>

In [ ]:
def reference_bins(values: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Bin edges for a PSI comparison, taken from the reference window only.

    Flatten `values`, cut it at the `n_bins - 1` interior quantiles, and return the edges
    with `-inf` first and `+inf` last, so the result has `n_bins + 1` entries on a reference
    with no ties. Collapse duplicate interior cut points, which leaves fewer bins rather than
    a zero-width one.

    Raise `ValueError` if `values` is empty, if any value is not finite, or if `n_bins < 2`.

    Returns: a 1-D float array of strictly increasing edges, `-inf` first and `+inf` last.

    Example:
        >>> reference_bins(np.array([0.0, 1.0, 2.0, 3.0]), 2)
        array([-inf, 1.5,  inf])
        >>> reference_bins(np.array([5.0, 5.0, 5.0, 5.0]), 4)   # every cut point ties
        array([-inf,   5.,  inf])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_reference_bins() -> None:
    got = reference_bins(np.array([0.0, 1.0, 2.0, 3.0]), 2)
    assert np.array_equal(got, np.array([-np.inf, 1.5, np.inf])), (
        f"expected [-inf, 1.5, inf] on the worked example, got {got}. One interior cut point "
        "for two bins, and the outer edges are infinite"
    )
    four = reference_bins(np.arange(101.0), 4)
    assert four.size == 5, f"n_bins=4 gives 5 edges on a tie-free reference, got {four.size}"
    assert np.isinf(four[0]) and four[0] < 0 and np.isinf(four[-1]) and four[-1] > 0, (
        "the first edge must be -inf and the last +inf, so no later reading can fall outside "
        "the bins; a finite outer edge silently drops the drift you are looking for"
    )
    assert np.allclose(four[1:-1], [25.0, 50.0, 75.0]), (
        f"the interior edges are the quartiles 25/50/75, got {four[1:-1]}. Equal-WIDTH bins "
        "would give 25/50/75 here too — try it on a skewed reference and they do not"
    )
    skew = reference_bins(np.array([0.0, 0.0, 0.0, 0.0, 100.0]), 2)
    assert np.isclose(skew[1], 0.0), (
        f"the median of [0,0,0,0,100] is 0.0, so that is the single interior cut; got "
        f"{skew[1]}. A value of 50.0 means you cut the RANGE in half instead of the MASS"
    )
    tied = reference_bins(np.array([5.0, 5.0, 5.0, 5.0]), 4)
    assert np.array_equal(tied, np.array([-np.inf, 5.0, np.inf])), (
        f"a reference with one repeated value has one distinct cut point; got {tied}. "
        "Duplicate edges make a zero-width bin, and the next line divides by its count"
    )
    assert np.all(np.diff(reference_bins(np.array([0.0, 0.0, 1.0, 9.0]), 6)) > 0), (
        "the edges must come back strictly increasing after the duplicates are collapsed"
    )
    for bad_v, bad_n in ((np.array([]), 4), (np.array([1.0, 2.0]), 1),
                         (np.array([1.0, 2.0]), 0), (np.array([1.0, np.nan]), 4),
                         (np.array([1.0, np.inf]), 4)):
        try:
            reference_bins(bad_v, bad_n)
        except ValueError:
            continue
        raise AssertionError(
            f"reference_bins({bad_v.tolist()}, {bad_n}) must raise ValueError")
    print("exercise 1 looks right — bins cut the reference's mass, not its range")

In [ ]:
_try("exercise 1", _check_reference_bins)

## 3. Exercise 2 — `psi()`, and what "no drift" measures

The population stability index compares two histograms over the same bins:

`PSI = sum over bins of (c_i - r_i) * ln(c_i / r_i)`

with `r_i` and `c_i` the *proportions* of the reference and the current window in bin `i`.
It is symmetric, it is zero when the two agree, and it grows when mass moves between bins.
An empty bin would take the logarithm of zero, so proportions are floored at `PSI_FLOOR`
before the ratio is taken.

Nothing about it needs a label, which is the whole reason this module can use it: module 7
left the plant with a one-way path out and no automatic way back, and built exactly this
statistic for exactly that reason. Here it takes its bin edges as an argument, because
section 4 needs a different set of them inside every duty regime.

<details><summary>💡 Hint 1 — what to think about</summary>

Two readings against four is not drift, so what must each histogram be divided by before
the two are compared? Then the empty bin: skip its term and you silence the loudest drift
there is; take its logarithm as it stands and the answer is no longer a number. And a value
sitting exactly on an interior edge — which of its two neighbouring bins owns it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate the edges, both samples and the floor before counting anything. Count each
flattened sample into the bins, sending a value on an edge to the bin above it. Divide each
count by the size of its OWN sample and raise both proportion arrays to at least the floor.
Then sum, bin by bin, the difference of the two proportions times the natural log of their
ratio, both taken current over reference as in the formula above. Drop either factor and
you have a one-sided divergence, which is not symmetric.
</details>

In [ ]:
def psi(reference: np.ndarray, current: np.ndarray, edges: np.ndarray,
        floor: float = PSI_FLOOR) -> float:
    """Population stability index between two samples over fixed bins.

    Bin `i` holds values in `[edges[i], edges[i+1])`. Both arrays are flattened. Convert the
    two counts to proportions of their own sample, floor both at `floor`, and sum
    `(c - r) * log(c / r)` over the bins.

    Raise `ValueError` if `edges` has fewer than 3 entries or is not strictly increasing, if
    either sample is empty, or if `floor` is not positive.

    Returns: a single float, zero when the two samples fill the bins identically.

    Example:
        >>> e = np.array([-np.inf, 0.5, np.inf])
        >>> round(psi(np.array([0.0, 1.0]), np.array([0.0, 1.0]), e), 9)
        0.0
        >>> round(psi(np.array([0.0, 1.0]), np.array([0.0, 0.0, 0.0, 1.0]), e), 4)
        0.2747
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_psi() -> None:
    e = np.array([-np.inf, 0.5, np.inf])
    same = psi(np.array([0.0, 1.0]), np.array([0.0, 1.0]), e)
    assert np.isclose(same, 0.0, atol=1e-9), (
        f"two identical samples have PSI 0.0, got {same!r}"
    )
    moved = psi(np.array([0.0, 1.0]), np.array([0.0, 0.0, 0.0, 1.0]), e)
    assert np.isclose(moved, 0.274653, atol=1e-5), (
        f"expected 0.274653 for 50/50 against 75/25, got {moved!r}. A value near 2.197 means "
        "you compared COUNTS rather than proportions — 2 readings against 4 is not drift. A "
        "value near 0.131 is the one-sided KL divergence, which is not symmetric"
    )
    swapped = psi(np.array([0.0, 0.0, 0.0, 1.0]), np.array([0.0, 1.0]), e)
    assert np.isclose(swapped, moved, atol=1e-9), (
        f"PSI is symmetric in its two samples: {swapped!r} != {moved!r}. An asymmetric "
        "answer means you divided by the wrong one somewhere"
    )
    scaled = psi(np.zeros(10), np.zeros(5000), e)
    assert np.isclose(scaled, 0.0, atol=1e-9), (
        f"two samples of very different size that fill the bins the same way have PSI 0, got "
        f"{scaled!r} — proportions, not counts"
    )
    empty_bin = psi(np.array([0.0, 1.0]), np.array([0.0, 0.0]), e)
    assert np.isfinite(empty_bin) and empty_bin > 1.0, (
        f"a bin the current window never fills is floored, not skipped and not infinite; got "
        f"{empty_bin!r}. inf or nan means log(0); a small number means you dropped the bin, "
        "and an emptied bin is the loudest drift there is"
    )
    three = psi(np.array([0.0, 1.0, 2.0]), np.array([0.0, 1.0, 2.0]),
                np.array([-np.inf, 0.5, 1.5, np.inf]))
    assert np.isclose(three, 0.0, atol=1e-9), "three bins must work as well as two"
    edge_case = psi(np.array([0.5, 0.5]), np.array([0.0, 0.0]), e)
    assert edge_case > 1.0, (
        "bin i is the HALF-OPEN interval [edges[i], edges[i+1]), so a value exactly on an "
        "interior edge belongs to the bin above it"
    )
    for bad in (np.array([-np.inf, np.inf]), np.array([0.0, 0.0, 1.0]),
                np.array([0.0, 1.0, 0.5])):
        try:
            psi(np.array([0.0]), np.array([0.0]), bad)
        except ValueError:
            continue
        raise AssertionError(f"edges={bad.tolist()} must raise ValueError")
    for bad_r, bad_c, bad_f in ((np.array([]), np.array([0.0]), PSI_FLOOR),
                                (np.array([0.0]), np.array([]), PSI_FLOOR),
                                (np.array([0.0]), np.array([0.0]), 0.0),
                                (np.array([0.0]), np.array([0.0]), -1e-6)):
        try:
            psi(bad_r, bad_c, e, bad_f)
        except ValueError:
            continue
        raise AssertionError(f"psi({bad_r.tolist()}, {bad_c.tolist()}, e, {bad_f}) must raise")
    print("exercise 2 looks right — proportions, a floor, and a symmetric answer")

In [ ]:
_try("exercise 2", _check_psi)

A PSI is a number with no units and no meaning until you know what this plant produces when
**nothing has changed**. The cell below measures that: split the reference window's units at
random into halves and score one against the other, many times, on both the health index and
the instrument self-test. The largest value it ever produces is the alert level used for the
rest of the notebook. No rule of thumb is typed anywhere in this lesson.

In [ ]:
def null_psi_band(reference: Window, n_bins: int = N_BINS, n_draws: int = NULL_DRAWS,
                  seed: int = SEED) -> float:
    """The largest PSI this plant produces when nothing has changed. Given to you.

    Splits the reference window's UNITS in half at random — not its rows — because the units
    are what a later window resamples. A row-level split would understate the noise and hand
    you an alert level that fires on nothing at all.
    """
    rng = np.random.default_rng(seed)
    n = reference.health.shape[0]
    worst = 0.0
    for _ in range(int(n_draws)):
        perm = rng.permutation(n)
        a, b = perm[: n // 2], perm[n // 2:]
        for stream in (reference.health, reference.check):
            left, right = stream[a].ravel(), stream[b].ravel()
            worst = max(worst, psi(left, right, reference_bins(left, n_bins)))
    return float(worst)


def _show_null_band() -> None:
    band = null_psi_band(REFERENCE)
    print(f"alert level, measured: the largest PSI over {NULL_DRAWS} random half-splits of "
          f"the reference")
    print(f"  fleet of {N_UNITS} units, {N_BINS} bins, both streams: {band:.5f}")
    print(f"  anything above {band:.5f} is something this plant does not do by chance.")


_try("null band", _show_null_band, needs=("exercise 1", "exercise 2"))

## 3b. Why the bins are quantiles

Module 7 built a PSI too, with `n_bins` **equal-width** bins spanning the commissioning
window's range. That is the right first version and the one most code uses. This module
factors the binning out into its own function for two reasons: section 4 needs a different
bin set *inside each duty regime*, and equal-width bins over a skewed feature put most of
the reference's mass into one or two bins, which is where a change in the tail goes to
hide. The next cell measures both rules on this fleet rather than arguing about them.

In [ ]:
def equal_width_bins(values: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Module 7's binning, carried forward so the two can be compared. Given to you."""
    v = np.asarray(values, dtype=float).ravel()
    cuts = np.linspace(v.min(), v.max(), int(n_bins) + 1)
    return np.concatenate(([-np.inf], cuts[1:-1], [np.inf]))


def _show_why_the_bins_are_quantiles() -> None:
    ref = REFERENCE.health
    print(f"  {'binning rule':<28}{'bins':>6}{'fullest bin':>13}{'empty bins':>12}"
          f"{'PSI, ageing':>13}{'PSI, steady':>13}")
    for label, edges in (("module 7, equal width", equal_width_bins(ref)),
                         ("module 8, equal mass", reference_bins(ref))):
        share = (np.bincount(np.searchsorted(edges[1:-1], ref.ravel(), side="right"),
                             minlength=edges.size - 1) / ref.size)
        print(f"  {label:<28}{edges.size - 1:>6}{share.max():>12.1%}"
              f"{int((share == 0).sum()):>12}"
              f"{psi(ref, LIVE['ageing'].health, edges):>13.4f}"
              f"{psi(ref, LIVE['steady'].health, edges):>13.4f}")
    print("\nboth rules see the ageing fleet and both stay quiet on the steady one, so the")
    print("choice is not about whether drift is visible. It is about resolution: equal-width")
    print("bins spend most of their range on a tail almost nothing occupies, and the fullest")
    print("bin column says how much of the reference that leaves in one place. Section 4")
    print("needs a bin set per duty regime, on three differently-shaped subsets, and the")
    print("equal-mass rule is the one that behaves the same way on all three.")


_try("why quantile bins", _show_why_the_bins_are_quantiles,
     needs=("exercise 1", "exercise 2"))

## 4. Exercise 3 — `conditional_psi()`

The plant runs its pumps harder in winter. The feature rises, the marginal PSI rises, and
nothing is wrong with a single machine. The duty point is not something you have to infer —
it is a setpoint, and the historian has it on every row. So ask the question again *inside*
each duty regime: cut the reference's readings at that regime with that regime's own bins,
score the current window's readings at that regime against them, and average over regimes
weighted by how much of the **reference** sat in each.

A drift that survives conditioning is a drift in the machines. A drift that vanishes was a
drift in the work. One rule has no sensible answer: a regime the reference never contains.
You cannot condition on a duty point you have never measured a healthy machine at, and
pretending otherwise is how a cold start becomes a fleet-wide alarm.

<details><summary>💡 Hint 1 — what to think about</summary>

Two choices decide this function. Inside one regime, whose readings set the bins? If it is
not the reference readings at that regime, the bins move with the thing you are measuring.
And whose mix sets the weights? The current window's mix is exactly what you are trying to
condition away, so it cannot also decide how much each regime counts.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check both value/regime pairs match in shape, then refuse any regime the current window
holds that the reference never did. For each regime present in both: build its bins from
its own reference readings with exercise 1, score its current readings with exercise 2,
and keep its reference count as the weight. Return the weighted average over the regimes
you actually scored; if there were none, raise.
</details>

In [ ]:
def conditional_psi(reference: np.ndarray, reference_regime: np.ndarray,
                    current: np.ndarray, current_regime: np.ndarray,
                    n_bins: int = N_BINS) -> float:
    """PSI computed inside each duty regime and averaged over the reference's regime mix.

    For every regime present in `reference_regime`: take that regime's reference readings,
    build `reference_bins` from them, and score that regime's current readings against them
    with `psi`. Weight each regime by its share of the REFERENCE readings — the current
    window's mix is the thing you are conditioning away, so it cannot also be the weights.
    Regimes with no readings in the current window are skipped and their weight redistributed.

    Raise `ValueError` if either array pair has mismatched shapes, if a regime appears in
    `current_regime` but not in `reference_regime`, or if no regime appears in both.

    Returns: a single float, on the same scale as `psi`.

    Example:
        >>> ref = np.array([0.0, 0.0, 1.0, 1.0]); reg = np.array([0, 0, 1, 1])
        >>> round(conditional_psi(ref, reg, ref, reg, 2), 9)   # nothing moved
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_conditional_psi() -> None:
    ref = np.array([0.0, 0.0, 1.0, 1.0])
    reg = np.array([0, 0, 1, 1])
    same = conditional_psi(ref, reg, ref, reg, 2)
    assert np.isclose(same, 0.0, atol=1e-9), f"nothing moved, so PSI is 0.0; got {same!r}"

    # A PURE regime change: within each regime the readings are identical, only the mix
    # moved. Conditioning must make it disappear.
    r0 = np.linspace(0.0, 1.0, 600)
    r1 = np.linspace(5.0, 6.0, 400)
    r_val = np.concatenate([r0, r1])
    r_reg = np.concatenate([np.zeros(600, int), np.ones(400, int)])
    c0 = np.linspace(0.0, 1.0, 120)
    c1 = np.linspace(5.0, 6.0, 900)
    c_val = np.concatenate([c0, c1])
    c_reg = np.concatenate([np.zeros(120, int), np.ones(900, int)])
    cond = conditional_psi(r_val, r_reg, c_val, c_reg, 3)
    marg = psi(r_val, c_val, reference_bins(r_val, 3))
    assert cond < 0.05 * marg, (
        f"a pure change of duty mix must vanish once you condition on duty: conditional "
        f"{cond:.4f} against marginal {marg:.4f}. If they are close you are binning the "
        "POOLED reference instead of each regime's own readings, or you never split at all"
    )

    # WHOSE quantiles. With a single duty point, conditional_psi IS psi over that regime's
    # reference bins, so this pins the bin source without pinning how you write it.
    ref_one = np.concatenate([np.linspace(0.0, 0.4, 500), np.linspace(0.9, 1.0, 100)])
    cur_one = np.concatenate([np.linspace(0.0, 0.1, 100), np.linspace(0.3, 1.0, 500)])
    flat = np.zeros(600, dtype=int)
    one = conditional_psi(ref_one, flat, cur_one, flat, 4)
    assert np.isclose(one, psi(ref_one, cur_one, reference_bins(ref_one, 4)), rtol=1e-9), (
        f"with one duty point the answer is psi over bins built from the REFERENCE readings "
        f"at that duty point; got {one:.4f}. Binning each regime's CURRENT readings instead "
        "makes the bins move with the drift, which is how you build a detector that reports "
        "nothing has happened"
    )

    # Weighting. Regime 0 is 60% of the reference and 12% of the current window, and its
    # readings have walked off the end of the scale. The answer is the reference-weighted
    # average of the two per-regime PSIs and nothing else.
    c0_far = np.linspace(90.0, 91.0, 120)
    c_far = np.concatenate([c0_far, c1])
    got = conditional_psi(r_val, r_reg, c_far, c_reg, 3)
    psi_0 = psi(r0, c0_far, reference_bins(r0, 3))
    psi_1 = psi(r1, c1, reference_bins(r1, 3))
    expected = (600 * psi_0 + 400 * psi_1) / 1000
    assert np.isclose(got, expected, rtol=1e-9), (
        f"expected {expected:.4f} — 0.6 of regime 0's PSI plus 0.4 of regime 1's — got "
        f"{got:.4f}. Weighting by the CURRENT mix gives about "
        f"{(120 * psi_0 + 900 * psi_1) / 1020:.4f} and buries the drift in the regime the "
        f"plant has stopped using; an unweighted mean gives about "
        f"{(psi_0 + psi_1) / 2:.4f}"
    )
    shifted = conditional_psi(r_val, r_reg, r_val + 3.0, r_reg, 3)
    assert shifted > 1.0, (
        f"shifting every reading within its own regime is exactly the drift conditioning "
        f"must NOT explain away; got {shifted:.4f}"
    )
    try:
        conditional_psi(r_val, r_reg, np.array([1.0]), np.array([7]), 3)
    except ValueError:
        pass
    else:
        raise AssertionError("a regime in the current window that the reference never saw "
                             "must raise ValueError, not be silently skipped")
    for bad in ((r_val, np.array([0, 0]), c_val, c_reg),
                (r_val, r_reg, c_val, np.array([1, 1]))):
        try:
            conditional_psi(*bad, 3)
        except ValueError:
            continue
        raise AssertionError("mismatched value and regime shapes must raise ValueError")
    print("exercise 3 looks right — each regime gets its own bins and the reference's weight")

In [ ]:
_try("exercise 3", _check_conditional_psi)

## 5. Exercise 4 — `drift_signature()`

Three numbers, none of which needs a label:

- **marginal** — has the feature moved at all?
- **conditional** — has it moved *within* a duty regime?
- **check** — has the instrument's own self-test moved?

The third is the one people leave out, and without it the problem is not merely hard but
*unidentifiable*: "every sensor reads half a unit high" and "every machine is half a unit
worse" produce exactly the same feature distribution. A check standard is what breaks the
tie, which is why the instrument reports one and why section 6 consults it first.

<details><summary>💡 Hint 1 — what to think about</summary>

The self-test is a different stream with its own centre and spread. Cut it with the health
index's bins and most of it falls into a bin or two, where a move is invisible. For the
conditional term, each window carries its own duty-point array: which one belongs beside
which readings?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

One field at a time. Marginal: exercise 2 on the two health arrays, over bins built from the
reference's health readings. Conditional: exercise 3 with each window's own health and
regime. Check: exercise 2 on the two check arrays, over bins built from the reference's
check readings. Median shift: current median minus reference median. Pass n_bins to every
bin set you build.
</details>

In [ ]:
def drift_signature(reference: Window, current: Window, n_bins: int = N_BINS
                    ) -> DriftSignature:
    """The three unsupervised statistics, plus the median shift for the report.

    * `marginal` — `psi` of `current.health` against `reference.health`, over bins built from
      the reference's health readings.
    * `conditional` — `conditional_psi` of the same two, using each window's regime array.
    * `check` — `psi` of `current.check` against `reference.check`, over bins built from the
      reference's CHECK readings. The health index's bins are the wrong bins for a stream
      that lives on a different scale.
    * `median_shift` — `median(current.health) - median(reference.health)`. Reported, never
      used for a decision: section 1 showed three windows that move it for three reasons.

    Returns: a `DriftSignature`. No labels are read anywhere in this function.

    Example:
        >>> sig = drift_signature(REFERENCE, REFERENCE)      # a window against itself
        >>> round(sig.marginal, 9), round(sig.check, 9), round(sig.median_shift, 9)
        (0.0, 0.0, 0.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_drift_signature() -> None:
    self_sig = drift_signature(REFERENCE, REFERENCE)
    for field in ("marginal", "conditional", "check"):
        got = getattr(self_sig, field)
        assert np.isclose(got, 0.0, atol=1e-9), (
            f"a window scored against itself has {field} 0.0, got {got!r}"
        )
    assert np.isclose(self_sig.median_shift, 0.0, atol=1e-12), (
        "median_shift is current minus reference, so a window against itself gives 0.0"
    )
    recal = drift_signature(REFERENCE, LIVE["recalibration"])
    ageing = drift_signature(REFERENCE, LIVE["ageing"])
    assert recal.check > 20 * ageing.check, (
        f"only the recalibration window moved the instrument self-test: check "
        f"{recal.check:.4f} against ageing's {ageing.check:.4f}. If they are similar you are "
        "scoring the CHECK readings against bins built from the HEALTH readings"
    )
    regime = drift_signature(REFERENCE, LIVE["regime"])
    assert regime.conditional < 0.35 * regime.marginal, (
        f"the regime window's drift is explained by duty: conditional "
        f"{regime.conditional:.4f} against marginal {regime.marginal:.4f}. If conditional is "
        "as large as marginal, check you passed each window's OWN regime array"
    )
    assert ageing.conditional > 0.5 * ageing.marginal, (
        f"the ageing window's drift survives conditioning — the duty mix did not move, the "
        f"machines did; got conditional {ageing.conditional:.4f} against marginal "
        f"{ageing.marginal:.4f}"
    )
    assert all(getattr(recal, f) >= 0.0 for f in ("marginal", "conditional", "check")), (
        "PSI is non-negative; a negative component means the two proportion arrays are "
        "swapped inside the sum"
    )
    print("exercise 4 looks right — three statistics, three different bin sets, no labels")

In [ ]:
_try("exercise 4", _check_drift_signature)

Run this. It is the whole diagnostic problem on one screen: five windows, three statistics,
and the median shift that cannot separate any of them.

In [ ]:
# Every demo from here on that scores a window against the reference needs these four.
_FOR_SIGNATURES = ("exercise 1", "exercise 2", "exercise 3", "exercise 4")


def _show_signatures() -> None:
    band = null_psi_band(REFERENCE)
    print(f"alert level (measured, section 3): {band:.5f}\n")
    print(f"  {'window':<18}{'marginal':>10}{'conditional':>13}{'check':>10}"
          f"{'median shift':>14}")
    for name in FLEET:
        if name == "reference":
            continue
        sig = drift_signature(REFERENCE, LIVE[name])
        print(f"  {name:<18}{sig.marginal:>10.4f}{sig.conditional:>13.4f}"
              f"{sig.check:>10.4f}{sig.median_shift:>+14.3f}")
    print("\nread the columns, not the rows. The median shift orders the windows in one way;")
    print("the three statistics disagree with that order and with each other, and the next")
    print("section is the rule that turns them into a name.")


_try("signature table", _show_signatures, needs=_FOR_SIGNATURES)

## 6. Exercise 5 — `diagnose_drift()`, and why the order of the tests is the lesson

Four outcomes, in the order they must be tested:

1. `marginal` below the alert level the plant measured — **none**. Stop.
2. `check` at or above it — **recalibration**. The instrument moved, so every other
   statistic is computed on numbers you cannot trust. This test comes *second*, before any
   reasoning about duty, for the same reason a lab re-checks the balance before re-running
   the assay.
3. `conditional` no more than `explained_fraction` of `marginal` — **regime**. Duty explains
   it; the machines are where you left them.
4. Otherwise — **ageing**. It is not the instrument and it is not the work, so it is the
   fleet, and section 9 will re-price rather than re-baseline.

The last window in the table does both at once — a turnaround that swapped transmitters
*and* changed the duty mix. Test duty first and you call it a regime change and condition
on a lie.

<details><summary>💡 Hint 1 — what to think about</summary>

The turnaround window satisfies two rules at once. Which of the two makes the other
statistic untrustworthy? That answers the order. Then look at the three boundaries: a
marginal exactly on the alert level, a check exactly on it, a conditional exactly at the
fraction of the marginal. The docstring says which side of the line each one falls on.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate first: a negative alert level, a fraction not strictly between 0 and 1, any
negative PSI component. Then test in the order the section lists and return at the first
rule that fires: quiet, then the instrument, then duty, and only then the fleet. Use the
`explained_fraction` you were handed, never the module constant, and return the exact
strings in `DIAGNOSES`.
</details>

In [ ]:
def diagnose_drift(signature: DriftSignature, psi_alert: float,
                   explained_fraction: float = EXPLAINED_FRACTION) -> str:
    """Name the drift from three unsupervised statistics. Returns one of `DIAGNOSES`.

    In order:
      * `signature.marginal < psi_alert`                              -> "none"
      * `signature.check >= psi_alert`                                -> "recalibration"
      * `signature.conditional <= explained_fraction * marginal`      -> "regime"
      * otherwise                                                     -> "ageing"

    Raise `ValueError` if `psi_alert` is negative, if `explained_fraction` is not strictly
    inside `(0, 1)`, or if any of the three PSI components is negative.

    Returns: one of "none", "recalibration", "regime", "ageing".

    Example:
        >>> diagnose_drift(DriftSignature(0.01, 0.01, 0.001, 0.0), 0.03)
        'none'
        >>> diagnose_drift(DriftSignature(1.20, 0.01, 0.001, 0.4), 0.03)
        'regime'
        >>> diagnose_drift(DriftSignature(1.20, 0.01, 0.900, 0.4), 0.03)
        'recalibration'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_diagnose_drift() -> None:
    quiet = DriftSignature(0.01, 0.01, 0.001, 0.0)
    assert diagnose_drift(quiet, 0.03) == "none", (
        "a marginal PSI below the alert level is no drift at all, whatever the other two "
        "columns say"
    )
    assert diagnose_drift(DriftSignature(0.03, 0.03, 0.001, 0.0), 0.03) == "ageing", (
        "the quiet test is `marginal < psi_alert`: a marginal exactly ON the alert level is "
        "drift, so this falls through to the last branch"
    )
    assert diagnose_drift(DriftSignature(1.2, 0.01, 0.001, 0.4), 0.03) == "regime", (
        "conditional is under half of marginal and the instrument is clean: duty explains it"
    )
    assert diagnose_drift(DriftSignature(1.2, 0.6, 0.001, 0.4), 0.03) == "regime", (
        "the regime test is `<=`: conditional exactly at explained_fraction * marginal is "
        "explained by duty"
    )
    assert diagnose_drift(DriftSignature(1.2, 0.61, 0.001, 0.4), 0.03) == "ageing", (
        "just past explained_fraction * marginal, duty no longer explains it, the instrument "
        "is clean, so the machines are worse"
    )
    assert diagnose_drift(DriftSignature(1.2, 1.2, 0.03, 0.4), 0.03) == "recalibration", (
        "the check test is `>=` the SAME alert level the marginal is judged against; a check "
        "exactly on it counts as moved"
    )
    # THE ORDERING CASE. Both the check and the duty test fire. A moved instrument outranks
    # a moved duty point, because conditioning on duty cannot repair a number read wrong.
    both = DriftSignature(marginal=2.0, conditional=0.5, check=0.9, median_shift=0.5)
    assert diagnose_drift(both, 0.03) == "recalibration", (
        f"this signature satisfies the regime rule (0.50 <= 0.5 * 2.00) AND the check rule "
        f"(0.90 >= 0.03). Testing duty first calls it 'regime' and conditions on readings "
        "the transmitter got wrong. The instrument is checked first"
    )
    assert diagnose_drift(DriftSignature(0.02, 0.02, 0.9, 0.0), 0.03) == "none", (
        "the quiet test comes before everything, including the instrument: if the feature "
        "has not moved there is nothing to fix, and a self-test wobble alone is a job for "
        "the metrology team, not a reason to touch the alarm"
    )
    assert diagnose_drift(DriftSignature(1.2, 0.9, 0.001, 0.4), 0.03,
                          explained_fraction=0.8) == "regime", (
        "explained_fraction is a parameter, not a constant baked into the comparison"
    )
    assert set(DIAGNOSES) == {"none", "recalibration", "regime", "ageing"}, (
        "return the exact strings in DIAGNOSES"
    )
    for bad_alert, bad_frac in ((-0.01, 0.5), (0.03, 0.0), (0.03, 1.0), (0.03, -0.5),
                                (0.03, 1.5)):
        try:
            diagnose_drift(quiet, bad_alert, bad_frac)
        except ValueError:
            continue
        raise AssertionError(
            f"psi_alert={bad_alert}, explained_fraction={bad_frac} must raise ValueError")
    for bad_sig in (DriftSignature(-0.1, 0.1, 0.1, 0.0), DriftSignature(0.1, -0.1, 0.1, 0.0),
                    DriftSignature(0.1, 0.1, -0.1, 0.0)):
        try:
            diagnose_drift(bad_sig, 0.03)
        except ValueError:
            continue
        raise AssertionError(f"a negative PSI component in {bad_sig} must raise ValueError")
    print("exercise 5 looks right — quiet, then the instrument, then duty, then the fleet")

In [ ]:
_try("exercise 5", _check_diagnose_drift)

Run this. It names every window, and then it prices the ordering mistake by re-running the
same rule with the duty test moved in front of the instrument test.

In [ ]:
def _wrong_order(signature: DriftSignature, psi_alert: float,
                 explained_fraction: float = EXPLAINED_FRACTION) -> str:
    """The same four rules with duty tested before the instrument. Given to you, to compare."""
    if signature.marginal < psi_alert:
        return "none"
    if signature.conditional <= explained_fraction * signature.marginal:
        return "regime"
    if signature.check >= psi_alert:
        return "recalibration"
    return "ageing"


TRUTH = {"steady": "none", "recalibration": "recalibration", "regime": "regime",
         "ageing": "ageing", "recal_and_regime": "recalibration"}


def _show_diagnoses() -> None:
    band = null_psi_band(REFERENCE)
    print(f"  {'window':<18}{'injected':>16}{'diagnosed':>16}{'duty tested first':>20}")
    disagree = 0
    for name in FLEET:
        if name == "reference":
            continue
        sig = drift_signature(REFERENCE, LIVE[name])
        mine = diagnose_drift(sig, band)
        theirs = _wrong_order(sig, band)
        disagree += mine != theirs
        flag = "" if mine == TRUTH[name] else "   <- MISSED"
        print(f"  {name:<18}{TRUTH[name]:>16}{mine:>16}{theirs:>20}{flag}")
    sig = drift_signature(REFERENCE, LIVE["recal_and_regime"])
    print(f"\nthe two orders disagree on {disagree} of {len(FLEET) - 1} windows.")
    print(f"on recal_and_regime the duty rule fires ({sig.conditional:.4f} <= "
          f"{EXPLAINED_FRACTION:.2f} x {sig.marginal:.4f} = "
          f"{EXPLAINED_FRACTION * sig.marginal:.4f}) and so does the instrument rule "
          f"({sig.check:.4f} >= {band:.5f}).")
    print("both are true. Only one of them is a reason to trust the other two statistics.")


_try("diagnoses", _show_diagnoses, needs=_FOR_SIGNATURES + ("exercise 5",))

## 7. Exercise 6 — `adjust_feature()`, the three fixes

Three drifts, three fixes, and each one undoes exactly one thing:

- **re-baseline** — re-zero each channel against its own instrument check. NIST's
  measurement handbook calls the check standard "a device for controlling the bias and
  long-term variability of the process once a baseline for these quantities has been
  established from historical data on the check standard", and that is what
  `baseline.check_level` is. It removes a transmitter offset and, by construction, nothing
  else: a unit whose self-test never moved is returned unchanged.
- **condition** — subtract the duty offsets measured on healthy reference data, so a
  reading taken at high duty is compared with what high duty looks like.
- **re-price** — leave the readings alone and move the threshold. That is exercise 8.

Each of the two feature fixes puts the readings back into the units the kick-off threshold
was derived in, so neither of them touches the threshold. Section 12 shows what happens
when you re-baseline against the *feature* instead of against the check, which is the
version most monitoring systems ship.

<details><summary>💡 Hint 1 — what to think about</summary>

Each fix undoes exactly one thing. Duty belongs to an hour and an instrument's shift belongs
to a unit, so what shape is the thing you subtract in each case? For the two fixes that
leave the readings alone, what goes wrong in section 9 if you hand back the caller's own
array? And which way does a channel that reads high have to move?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate everything first. `none` and `reprice` return a float copy. `condition` looks up
each reading's own regime in the baseline's offsets and subtracts that, reading by reading.
`rebaseline` takes each unit's median self-test reading minus the kick-off check level as
that unit's shift and subtracts it from the whole of that unit's row — a median, so one
absurd self-test reading cannot move a healthy channel. Never write into `health`.
</details>

In [ ]:
def adjust_feature(fix: str, health: np.ndarray, regime: np.ndarray, check: np.ndarray,
                   baseline: Baseline) -> np.ndarray:
    """Return the feature the alarm rule should see under `fix`. Never modifies `health`.

    * `"none"` and `"reprice"` — the readings come back unchanged, as a copy. Re-pricing
      moves the threshold, not the readings.
    * `"rebaseline"` — subtract, from every reading of unit `u`, that unit's own instrument
      shift: `median(check[u]) - baseline.check_level`.
    * `"condition"` — subtract, from every reading, `baseline.regime_offset[r]` for that
      reading's own regime `r`.

    Raise `ValueError` if `fix` is not in `FIXES`, if `health` is not 2-D, if `regime` does
    not have the same shape as `health`, if `check` does not have one row per unit, or if
    `regime` holds an index `baseline.regime_offset` does not cover.

    Returns: a new float array shaped like `health`.

    Example:
        >>> b = Baseline(0.9, (0.0, 0.5), 1.0, 2.0, Prices(1.0, 10.0, 2.0))
        >>> h = np.array([[1.0, 2.0, 3.0]]); r = np.array([[0, 1, 1]])
        >>> c = np.array([[1.4, 1.4]])                 # this channel reads 0.4 high
        >>> adjust_feature("condition", h, r, c, b)
        array([[1. , 1.5, 2.5]])
        >>> adjust_feature("rebaseline", h, r, c, b)
        array([[0.6, 1.6, 2.6]])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_adjust_feature() -> None:
    b = Baseline(0.9, (0.0, 0.5), 1.0, 2.0, Prices(1.0, 10.0, 2.0))
    h = np.array([[1.0, 2.0, 3.0]])
    r = np.array([[0, 1, 1]])
    c = np.array([[1.4, 1.4]])
    for inert in ("none", "reprice"):
        got = adjust_feature(inert, h, r, c, b)
        assert np.allclose(got, h), (
            f"{inert!r} leaves the readings alone, got {got}. Re-pricing moves the "
            "THRESHOLD; a fix that did both would make the matrix in section 9 a comparison "
            "of one thing with itself"
        )
        assert got is not h, (
            f"{inert!r} must return a copy: the caller's array is shared with every other "
            "fix in the matrix and mutating it corrupts the rest of the row"
        )
    cond = adjust_feature("condition", h, r, c, b)
    assert np.allclose(cond, [[1.0, 1.5, 2.5]]), (
        f"expected [[1.0, 1.5, 2.5]] — regime 1 carries an offset of 0.5 — got {cond}. Using "
        "one offset per UNIT instead of one per reading gives [[0.67, 1.67, 2.67]]; duty is "
        "a property of the hour"
    )
    reb = adjust_feature("rebaseline", h, r, c, b)
    assert np.allclose(reb, [[0.6, 1.6, 2.6]]), (
        f"this channel's self-test reads 1.4 against a kick-off level of 1.0, so every "
        f"reading drops by 0.4; expected [[0.6, 1.6, 2.6]], got {reb}. Adding the shift "
        "instead of subtracting it doubles the error you were trying to remove"
    )
    clean = adjust_feature("rebaseline", h, r, np.array([[1.0, 1.0]]), b)
    assert np.allclose(clean, h), (
        f"a unit whose self-test never moved must come back UNCHANGED, got {clean}. This is "
        "the property that makes re-baselining the wrong answer to ageing: it can only undo "
        "what the instrument did"
    )
    two = np.array([[1.0, 2.0, 3.0], [5.0, 6.0, 7.0]])
    per_unit = adjust_feature("rebaseline", two, np.zeros_like(two, dtype=int),
                              np.array([[1.4, 1.4], [1.0, 1.0]]), b)
    assert np.allclose(per_unit, [[0.6, 1.6, 2.6], [5.0, 6.0, 7.0]]), (
        f"the shift is PER UNIT: unit 0 moved and unit 1 did not; got {per_unit}. A single "
        "pooled median over the whole fleet moves both by the average and corrupts the "
        "channel that was working"
    )
    skew = adjust_feature("rebaseline", h, r, np.array([[1.0, 1.0, 1.0, 9.0]]), b)
    assert np.allclose(skew, h), (
        f"the shift is the MEDIAN of the self-test readings, so one absurd reading does not "
        f"move it; got {skew}. A mean gives 3.0 and shifts a healthy channel by 2.0"
    )
    untouched = np.array([[1.0, 2.0, 3.0]])
    adjust_feature("rebaseline", untouched, r, c, b)
    assert np.allclose(untouched, [[1.0, 2.0, 3.0]]), "do not modify the caller's array"
    for bad in (("polish", h, r, c), ("none", np.array([1.0, 2.0]), np.array([0, 1]),
                                      np.array([[1.0]])),
                ("condition", h, np.array([[0, 1]]), c),
                ("condition", h, np.array([[0, 1, 5]]), c),
                ("rebaseline", h, r, np.array([[1.0], [1.0]]))):
        try:
            adjust_feature(*bad, b)
        except ValueError:
            continue
        raise AssertionError(f"adjust_feature({bad[0]!r}, ...) must raise ValueError")
    print("exercise 6 looks right — one shift per instrument, one offset per hour, and a copy")

In [ ]:
_try("exercise 6", _check_adjust_feature)

Run this. It applies each fix to each window and prints what it does to the median health
index. Nothing is scored yet; the point is that each feature fix is a near no-op on the
windows it was not meant for, so the matrix in section 9 compares responses and not
side effects.

In [ ]:
def _show_what_the_fixes_remove() -> None:
    base = float(np.median(REFERENCE.health))
    print(f"reference median health: {base:.3f}\n")
    print(f"  {'window':<18}" + "".join(f"{f:>14}" for f in FIXES))
    for name in FLEET:
        if name == "reference":
            continue
        win = LIVE[name]
        row = [float(np.median(adjust_feature(f, win.health, win.regime, win.check, BASELINE)))
               for f in FIXES]
        print(f"  {name:<18}" + "".join(f"{v - base:>+14.3f}" for v in row))
    print("\n(each cell is the median of the adjusted feature minus the reference median)")
    print("each feature fix flattens exactly one row and leaves the others where they were.")
    print("Neither of them touches the ageing row at all, and that is the point: the fleet")
    print("really is failing more often, and no amount of arithmetic on the readings will")
    print("make that untrue. Section 9 prices what happens when you try.")


_try("what the fixes remove", _show_what_the_fixes_remove, needs=("exercise 6",))

## 8. Exercise 7 — `estimate_prices()`, the audit that closes the loop

Module 1 ended with a handover note carrying a threshold and the three prices it came from,
and the instruction to re-derive whenever a price changed. Nobody ever does, because nobody
collects the prices. The **alarm audit** is the collection: one line for every alarm that
was inspected and what the job cost, and one line for every failure that arrived without an
alarm.

This is the one place in the module where a label reaches you, and it is worth being exact
about why that does not contradict module 7. The diode blocks the *automatic* return path,
so the detector cannot be supervised. The audit comes back through people, in weeks, a few
dozen rows at a time. That is useless for training and sufficient for pricing.

The trap is the class with almost no rows. A no-fault-found call-out rarely reaches the
ledger, so `false_alarm` has a handful of entries and averaging them is worse than useless.
A class below `min_observations` keeps the kick-off guess **and says so**.

<details><summary>💡 Hint 1 — what to think about</summary>

What should a class with too few lines report: a number, or the guess plus an admission that
it is one? What does a class with no lines at all turn into if you average nothing, and what
does a zero price for a missed failure do to the threshold? And invoices are right-skewed:
is it the typical invoice or the average one that makes count times price equal the bill?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate `min_observations` and every entry first. Walk `OUTCOMES` in order — `Prices` has
the same field order. For each outcome count its entries; if the count reaches the minimum
use the mean cost, otherwise keep the fallback's price for that field and add the outcome's
name to `still_guessed`. The counts always report what the audit holds, used or not.
</details>

In [ ]:
TRUE_PLANNED = 11_800.0
TRUE_UNPLANNED = 96_500.0
TRUE_FALSE_ALARM = 5_400.0
BOOKED_FALSE_ALARM_RATE = 0.10   # share of no-fault call-outs that ever reach the ledger


def generate_audit(window: Window, baseline: Baseline, seed: int = SEED,
                   lead_hours: int = LEAD_HOURS) -> tuple:
    """The alarm audit as a plant keeps it, not as a data scientist would like it. Given.

    One entry per alarm that was inspected, priced off the invoice; one per failure that
    arrived with no alarm or too late to act on. False alarms are under-booked on purpose:
    an hour spent finding nothing is rarely written up, which is exactly why the price of a
    false alarm is the price a plant is least able to tell you.
    """
    rng = np.random.default_rng(seed + 1)
    entries = []
    for run in window.runs:
        row = window.health[run.unit, :run.end_hour]
        hit = np.flatnonzero(row >= baseline.threshold)
        alarm = int(hit[0]) if hit.size else -1
        if run.kind == FAILURE:
            if alarm >= 0 and run.end_hour - alarm >= lead_hours:
                entries.append(AuditEntry(run.unit, alarm, "planned",
                                          float(rng.lognormal(np.log(TRUE_PLANNED), 0.22))))
            else:
                entries.append(AuditEntry(run.unit, run.end_hour, "unplanned",
                                          float(rng.lognormal(np.log(TRUE_UNPLANNED), 0.30))))
        elif alarm >= 0 and rng.random() < BOOKED_FALSE_ALARM_RATE:
            entries.append(AuditEntry(run.unit, alarm, "false_alarm",
                                      float(rng.lognormal(np.log(TRUE_FALSE_ALARM), 0.25))))
    return tuple(entries)


# Two windows of history: the one the model was fitted on and the quiet year after it.
AUDIT = (generate_audit(REFERENCE, BASELINE)
         + generate_audit(FLEET["steady"], BASELINE, seed=SEED + 7))

In [ ]:
def estimate_prices(entries: Sequence[AuditEntry], fallback: Prices,
                    min_observations: int = MIN_OBSERVATIONS) -> PriceEstimate:
    """Estimate the three prices from the audit, keeping the guess where the audit is thin.

    For each outcome in `OUTCOMES` — which is in the same order as the fields of `Prices` —
    take the mean `cost` of the entries carrying it. Use that mean only when the outcome has
    at least `min_observations` entries; otherwise keep the matching price from `fallback`
    and record the outcome's name in `still_guessed`, in `OUTCOMES` order.

    An outcome with no entries at all must come back as the FALLBACK, never as zero: a
    missed failure priced at zero makes "never alarm" the cost-optimal policy.

    Raise `ValueError` if an entry carries an outcome not in `OUTCOMES`, if a cost is
    negative or not finite, or if `min_observations` is below 1.

    Returns: a `PriceEstimate` carrying the prices, the three counts in `OUTCOMES` order,
    and the tuple of outcome names still resting on the kick-off guess.

    Example:
        >>> guess = Prices(10.0, 100.0, 5.0)
        >>> rows = [AuditEntry(0, 1, "planned", 20.0), AuditEntry(1, 2, "planned", 30.0)]
        >>> estimate_prices(rows, guess, min_observations=2)
        PriceEstimate(prices=Prices(planned=25.0, unplanned=100.0, false_alarm=5.0), \
counts=(2, 0, 0), still_guessed=('unplanned', 'false_alarm'))
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_estimate_prices() -> None:
    guess = Prices(10.0, 100.0, 5.0)
    rows = [AuditEntry(0, 1, "planned", 20.0), AuditEntry(1, 2, "planned", 30.0)]
    got = estimate_prices(rows, guess, min_observations=2)
    assert np.isclose(got.prices.planned, 25.0), (
        f"the mean of 20 and 30 is 25.0, got {got.prices.planned!r}"
    )
    assert np.isclose(got.prices.unplanned, 100.0) and np.isclose(got.prices.false_alarm, 5.0), (
        f"an outcome the audit never saw keeps the kick-off guess, got {got.prices}. A zero "
        "here prices a missed failure at nothing and makes 'never alarm' optimal"
    )
    assert got.counts == (2, 0, 0), (
        f"counts are (planned, unplanned, false_alarm), got {got.counts}"
    )
    assert got.still_guessed == ("unplanned", "false_alarm"), (
        f"still_guessed names the fallen-back outcomes in OUTCOMES order, got "
        f"{got.still_guessed}. Reporting the estimate without it lets a number nobody "
        "measured be read as one somebody did"
    )
    thin = estimate_prices(rows, guess, min_observations=3)
    assert np.isclose(thin.prices.planned, 10.0) and thin.still_guessed == OUTCOMES, (
        f"two entries is below a minimum of three, so planned falls back too; got "
        f"{thin.prices} / {thin.still_guessed}"
    )
    assert thin.counts == (2, 0, 0), (
        "the counts report what the audit HOLDS, not what was used; a class can have two "
        "entries and still be a guess"
    )
    exact = estimate_prices(rows, guess, min_observations=2)
    assert np.isclose(exact.prices.planned, 25.0), (
        "the test is `>=` min_observations: exactly enough entries is enough"
    )
    skewed = estimate_prices(
        [AuditEntry(i, 0, "unplanned", c)
         for i, c in enumerate((10.0, 10.0, 10.0, 10.0, 60.0))], guess, 5)
    assert np.isclose(skewed.prices.unplanned, 20.0), (
        f"the MEAN of 10, 10, 10, 10 and 60 is 20.0; got {skewed.prices.unplanned!r}. A "
        "median gives 10.0 — invoices are right-skewed and expected_cost multiplies this "
        "number by a count, so the mean is the one that makes the total right"
    )
    mixed = estimate_prices(
        rows + [AuditEntry(2, 3, "unplanned", 900.0), AuditEntry(3, 4, "false_alarm", 7.0)],
        guess, min_observations=1)
    assert np.isclose(mixed.prices.unplanned, 900.0) and mixed.still_guessed == (), (
        f"with a minimum of one every class is measured; got {mixed.prices} / "
        f"{mixed.still_guessed}"
    )
    order = estimate_prices([AuditEntry(0, 0, "false_alarm", 3.0)], guess, 1)
    assert np.isclose(order.prices.false_alarm, 3.0) and np.isclose(order.prices.planned, 10.0), (
        f"OUTCOMES is ('planned', 'unplanned', 'false_alarm') and Prices has the same field "
        f"order; got {order.prices}. Zipping the two in any other order silently swaps the "
        "price of a call-out with the price of an overhaul"
    )
    empty = estimate_prices([], guess, 1)
    assert empty.prices == guess and empty.counts == (0, 0, 0), (
        f"an empty audit returns the kick-off guess untouched, got {empty}"
    )
    for bad in ([AuditEntry(0, 0, "scrapped", 1.0)], [AuditEntry(0, 0, "planned", -1.0)],
                [AuditEntry(0, 0, "planned", float("nan"))]):
        try:
            estimate_prices(bad, guess, 1)
        except ValueError:
            continue
        raise AssertionError(f"{bad} must raise ValueError")
    for bad_min in (0, -1):
        try:
            estimate_prices(rows, guess, bad_min)
        except ValueError:
            continue
        raise AssertionError(f"min_observations={bad_min} must raise ValueError")
    print("exercise 7 looks right — a thin class keeps its guess and admits to it")

In [ ]:
_try("exercise 7", _check_estimate_prices)

Run this. It is the kick-off whiteboard against the plant's own invoices.

In [ ]:
def _show_the_audit() -> None:
    est = estimate_prices(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    print(f"audit over {len(REFERENCE.runs) + len(FLEET['steady'].runs)} unit-windows of "
          f"history: {len(AUDIT)} lines")
    print(f"  {'price':<14}{'kick-off':>12}{'audited':>12}{'lines':>8}{'ratio':>9}")
    for name, guess, measured, n in zip(OUTCOMES, KICKOFF, est.prices, est.counts):
        mark = "  (still a guess)" if name in est.still_guessed else ""
        print(f"  {name:<14}{guess:>12,.0f}{measured:>12,.0f}{n:>8}"
              f"{measured / guess:>9.2f}{mark}")
    print(f"\nstill resting on the kick-off number: "
          f"{', '.join(est.still_guessed) if est.still_guessed else 'nothing'}")
    ratio_guess = KICKOFF.unplanned / KICKOFF.planned
    ratio_real = est.prices.unplanned / est.prices.planned
    print(f"the ratio that sets the threshold — unplanned over planned — was guessed at "
          f"{ratio_guess:.1f} and measures {ratio_real:.1f}.")


_try("the audit", _show_the_audit, needs=("exercise 7",))

## 9. Exercise 8 — `apply_fix()`, and the cost of the wrong fix

Now score them. `apply_fix` produces the feature the alarm rule sees and the threshold it
uses, then counts the matrix and prices it. Only one of the four fixes moves the threshold,
and it re-derives it on the **history** half of the fleet, never on the half it is about to
be scored on — a threshold fitted to the same rows it is graded on flatters every fix
equally and hides the whole point of the table.

<details><summary>💡 Hint 1 — what to think about</summary>

Two fleets arrive: one to fit on and one to be scored on. Which of the four fixes is allowed
to move the threshold, which fleet does it fit on, and at which prices: the ones frozen in
the baseline, or the ones you were handed? The cost at the end asks the same question about
prices again.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Adjust the window's feature with exercise 6. If the fix is `reprice`, adjust the history
window the same way and take the threshold from the given sweep on that history at the
prices passed in; for any other fix keep the baseline's threshold. Count the adjusted window
at that threshold with the lead time you were given, price the counts at the prices passed
in, and return the four fields together.
</details>

In [ ]:
def apply_fix(fix: str, window: Window, history: Window, baseline: Baseline, prices: Prices,
              thresholds: np.ndarray = THRESHOLDS, lead_hours: int = LEAD_HOURS) -> FixResult:
    """Apply one fix to one window and price the result.

    * the feature comes from
      `adjust_feature(fix, window.health, window.regime, window.check, baseline)`;
    * the threshold is `baseline.threshold` for every fix EXCEPT `"reprice"`, which re-derives
      it with `cost_optimal_threshold` on the history window adjusted the same way, at
      `prices`;
    * the counts come from `alarm_counts` on the adjusted window at that threshold, and the
      cost from `expected_cost` at `prices` — the audited prices, not `baseline.prices`.

    Raise `ValueError` for a fix outside `FIXES` (`adjust_feature` already does) or an empty
    `thresholds` (`cost_optimal_threshold` already does).

    Returns: a `FixResult` carrying the fix, the threshold it used, the counts and the cost.

    Example:
        >>> b = Baseline(1.0, (0.0,), 1.0, 2.0, Prices(1.0, 100.0, 5.0))
        >>> h = np.array([[1.0, 1.0, 3.0, 3.0], [1.0, 1.0, 1.0, 1.0]])
        >>> w = Window("w", h, np.zeros_like(h, dtype=int), np.ones((2, 2)),
        ...            (Run(0, 4, FAILURE), Run(1, 4, CENSORED)))
        >>> apply_fix("none", w, w, b, Prices(1.0, 100.0, 5.0), np.array([2.0]), 2)
        FixResult(fix='none', threshold=2.0, counts=Counts(tp=1, fn=0, fp=0, tn=1), cost=1.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_apply_fix() -> None:
    b = Baseline(1.0, (0.0,), 1.0, 2.0, Prices(1.0, 100.0, 5.0))
    h = np.array([[1.0, 1.0, 3.0, 3.0], [1.0, 1.0, 1.0, 1.0]])
    w = Window("w", h, np.zeros_like(h, dtype=int), np.ones((2, 2)),
               (Run(0, 4, FAILURE), Run(1, 4, CENSORED)))
    prices = Prices(1.0, 100.0, 5.0)
    got = apply_fix("none", w, w, b, prices, np.array([2.0]), 2)
    assert got == FixResult("none", 2.0, Counts(1, 0, 0, 1), 1.0), (
        f"expected FixResult('none', 2.0, Counts(1, 0, 0, 1), 1.0), got {got}"
    )
    for inert in ("none", "rebaseline", "condition"):
        r = apply_fix(inert, w, w, b, prices, np.array([2.0, 3.5]), 2)
        assert np.isclose(r.threshold, b.threshold), (
            f"{inert!r} moves the FEATURE and leaves the threshold at baseline.threshold "
            f"({b.threshold}); got {r.threshold}. Re-deriving under every fix makes the "
            "matrix in the next cell a comparison of one thing with itself"
        )
        assert r.fix == inert, f"the result carries the fix it was asked for, got {r.fix!r}"
    # reprice must actually move, and must read the prices it is HANDED. This second toy
    # has a noisy healthy unit, so the low threshold buys a catch AND a false alarm.
    noisy = np.array([[1.0, 1.0, 3.0, 3.0], [2.5, 2.5, 2.5, 2.5]])
    w2 = Window("w2", noisy, np.zeros_like(noisy, dtype=int), np.ones((2, 2)), w.runs)
    grid = np.array([2.0, 3.5])
    loud = apply_fix("reprice", w2, w2, b, Prices(1.0, 100.0, 5.0), grid, 2)
    assert np.isclose(loud.threshold, 2.0), (
        f"a missed failure costs 100 and the false alarm costs 5, so the sweep takes 2.0 and "
        f"buys the catch; got {loud.threshold}"
    )
    quiet = apply_fix("reprice", w2, w2, b, Prices(1.0, 1.0, 500.0), grid, 2)
    assert np.isclose(quiet.threshold, 3.5), (
        f"the SAME window, re-priced: a false alarm at 500 against a failure at 1 takes the "
        f"threshold to 3.5; got {quiet.threshold}. A threshold that does not move with the "
        "prices means you swept at baseline.prices instead of the prices you were handed"
    )
    priced = apply_fix("none", w2, w2, b, Prices(7.0, 11.0, 13.0), grid, 2)
    assert np.isclose(priced.cost, 7.0 + 13.0), (
        f"one catch at 7 and one false alarm at 13 is 20; got {priced.cost}. The cost uses "
        "the prices passed in, not baseline.prices — the audit is the whole point"
    )
    # the history is a DIFFERENT fleet from the one being scored
    past_h = np.array([[1.0, 1.0, 1.0, 1.0], [5.0, 5.0, 5.0, 5.0]])
    past = Window("past", past_h, np.zeros_like(past_h, dtype=int), np.ones((2, 2)),
                  (Run(0, 4, FAILURE), Run(1, 4, CENSORED)))
    split = apply_fix("reprice", w, past, b, Prices(1.0, 100.0, 5.0), np.array([2.0, 6.0]), 2)
    assert np.isclose(split.threshold, 6.0), (
        f"in the HISTORY window the failing unit never alarms and the censored one always "
        f"does, so the cheap threshold there is 6.0; got {split.threshold}. Re-deriving on "
        "`window` instead of `history` fits the threshold to the rows it is then scored on"
    )
    late = apply_fix("none", w, w, b, prices, np.array([2.0]), 3)
    assert late.counts == Counts(0, 1, 0, 1), (
        f"lead_hours must reach alarm_counts: the alarm lands at hour 2 and the run ends at "
        f"4, so three hours of warning was not available; got {late.counts}"
    )
    for bad_fix, bad_thr in (("polish", np.array([2.0])), ("reprice", np.array([]))):
        try:
            apply_fix(bad_fix, w, w, b, prices, bad_thr, 2)
        except ValueError:
            continue
        raise AssertionError(f"apply_fix({bad_fix!r}, ..., {bad_thr}) must raise ValueError")
    print("exercise 8 looks right — one fix moves the threshold, and it looks at history")

In [ ]:
_try("exercise 8", _check_apply_fix)

## 10. Exercise 9 — `respond_to_drift()`, end to end

One function: look at the window, name the drift, apply the fix that belongs to that name,
and price the result. The mapping is the lesson in four lines:

| diagnosis | fix | because |
|---|---|---|
| `none` | `none` | nothing moved; a change now is a change you cannot attribute |
| `recalibration` | `rebaseline` | the instrument moved, so re-zero it against its check |
| `regime` | `condition` | the work moved, so compare like with like |
| `ageing` | `reprice` | the fleet moved, so the base rate moved, and the base rate is an input to the cost function |

<details><summary>💡 Hint 1 — what to think about</summary>

Every piece already exists; this exercise is wiring. Which of your own parameters does each
of the three calls need? A caller who passes a different `n_bins` or `explained_fraction`
sees no difference if one of your calls quietly uses the module constant instead, and that
is the mistake to avoid.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build the signature against the reference with your `n_bins`, diagnose it with your
`psi_alert` and `explained_fraction`, look the diagnosis up in the table above to get its
fix, then apply that fix with the history, baseline, prices, thresholds and lead time you
were given. Return the diagnosis, the fix and the priced result together.
</details>

In [ ]:
def respond_to_drift(window: Window, history: Window, reference: Window, baseline: Baseline,
                     prices: Prices, psi_alert: float, thresholds: np.ndarray = THRESHOLDS,
                     lead_hours: int = LEAD_HOURS, n_bins: int = N_BINS,
                     explained_fraction: float = EXPLAINED_FRACTION) -> Response:
    """Diagnose the drift in `window` against `reference`, apply its fix, and price it.

    Three calls you have already written, and the mapping between them:
      * `drift_signature(reference, window, n_bins)`
      * `diagnose_drift(signature, psi_alert, explained_fraction)`
      * `"none" -> "none"`, `"recalibration" -> "rebaseline"`, `"regime" -> "condition"`,
        `"ageing" -> "reprice"`
      * `apply_fix(fix, window, history, baseline, prices, thresholds, lead_hours)`

    Returns: a `Response` carrying the diagnosis, the fix it implies and the `FixResult`.

    Example:
        >>> r = respond_to_drift(LIVE["steady"], HISTORY["steady"], REFERENCE, BASELINE,
        ...                      KICKOFF, psi_alert=1e9)     # an alert nothing can reach
        >>> r.diagnosis, r.fix, r.result.threshold == BASELINE.threshold
        ('none', 'none', True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_respond_to_drift() -> None:
    unreachable = respond_to_drift(LIVE["steady"], HISTORY["steady"], REFERENCE, BASELINE,
                                   KICKOFF, psi_alert=1e9)
    assert unreachable.diagnosis == "none" and unreachable.fix == "none", (
        f"with an alert level nothing can reach, every window is quiet; got "
        f"{unreachable.diagnosis!r} / {unreachable.fix!r}"
    )
    assert np.isclose(unreachable.result.threshold, BASELINE.threshold), (
        "the 'none' fix leaves the kick-off threshold exactly where it was"
    )
    band = null_psi_band(REFERENCE)
    expected = {"steady": ("none", "none"), "recalibration": ("recalibration", "rebaseline"),
                "regime": ("regime", "condition"), "ageing": ("ageing", "reprice")}
    for name, (want_d, want_f) in expected.items():
        got = respond_to_drift(LIVE[name], HISTORY[name], REFERENCE, BASELINE, KICKOFF, band)
        assert got.diagnosis == want_d, (
            f"the {name} window is a {want_d} drift, diagnosed {got.diagnosis!r}"
        )
        assert got.fix == want_f, (
            f"a {want_d} diagnosis calls for {want_f!r}, got {got.fix!r}. Wiring the four "
            "names to the wrong four fixes is the mistake this whole module exists to "
            "prevent, and section 9's matrix prices every one of them"
        )
        assert got.result.fix == got.fix, "the FixResult must carry the fix that was applied"
    aged = respond_to_drift(LIVE["ageing"], HISTORY["ageing"], REFERENCE, BASELINE, KICKOFF,
                            band)
    assert not np.isclose(aged.result.threshold, BASELINE.threshold), (
        f"re-pricing the ageing window must MOVE the threshold off {BASELINE.threshold}; got "
        f"{aged.result.threshold}. If it did not move, apply_fix is not re-deriving on the "
        "history window"
    )
    for name in ("recalibration", "regime"):
        got = respond_to_drift(LIVE[name], HISTORY[name], REFERENCE, BASELINE, KICKOFF, band)
        assert np.isclose(got.result.threshold, BASELINE.threshold), (
            f"the {name} response is a FEATURE fix and leaves the threshold alone; got "
            f"{got.result.threshold}"
        )
    print("exercise 9 looks right — one call from a window to a priced response")

In [ ]:
_try("exercise 9", _check_respond_to_drift)

## 11. The matrix

Four windows, four fixes, all sixteen priced at the audited prices, with each fix scored on
the live half of a fleet whose history it never saw. The recommended fix is starred.

In [ ]:
def _fix_matrix(prices: Prices) -> dict:
    """Every fix against every window, priced. Given to you — it is four nested calls."""
    out = {}
    for name in ("steady", "recalibration", "regime", "ageing"):
        out[name] = {f: apply_fix(f, LIVE[name], HISTORY[name], BASELINE, prices)
                     for f in FIXES}
    return out


def _show_matrix() -> None:
    band = null_psi_band(REFERENCE)
    audited = estimate_prices(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    matrix = _fix_matrix(audited.prices)
    print(f"cost of each response, at the audited prices "
          f"({', '.join(f'{p:,.0f}' for p in audited.prices)}):\n")
    print(f"  {'window':<16}" + "".join(f"{f:>15}" for f in FIXES) + "    chosen")
    worst_ratio, worst_line = 0.0, ""
    for name, row in matrix.items():
        chosen = respond_to_drift(LIVE[name], HISTORY[name], REFERENCE, BASELINE,
                                  audited.prices, band)
        cells = []
        for f in FIXES:
            mark = "*" if f == chosen.fix else " "
            cells.append(f"{row[f].cost:>14,.0f}{mark}")
        print(f"  {name:<16}" + "".join(cells) + f"    {chosen.fix}")
        for f in FIXES:
            ratio = row[f].cost / chosen.result.cost
            if ratio > worst_ratio:
                worst_ratio, worst_line = ratio, f"{f} on {name}"
    print("\n  * = the fix this window's own diagnosis asked for.")
    print(f"\nthe recommended fix is the cheapest cell in every row. The most expensive "
          f"mistake\nin the table is {worst_line}, at {worst_ratio:.2f}x the cost of the "
          f"right answer.")
    aged = matrix["ageing"]
    print(f"on the ageing fleet, conditioning on duty costs "
          f"{aged['condition'].cost / aged['reprice'].cost:.2f}x what re-pricing costs and "
          f"gives up\n{aged['condition'].counts.fn - aged['reprice'].counts.fn} more "
          f"failures. Nothing is wrong with the arithmetic. It is the wrong arithmetic.")


_try("the matrix", _show_matrix, needs=tuple(_EXERCISES))   # every exercise feeds it

One row of that table deserves a second look. On the **regime** window re-pricing lands on
exactly the same counts as conditioning — it raises the threshold until the duty-inflated
readings stop alarming, which on a fleet running mostly at high duty is nearly the same
thing. The two answers are not equivalent, and the next cell shows why: it prints how much
of each window was taken at high duty, then takes each fix's threshold and asks what it
would do when the parallel train comes back and duty returns to normal.

In [ ]:
def _show_the_regime_trap() -> None:
    audited = estimate_prices(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    row = _fix_matrix(audited.prices)["regime"]
    later = LIVE["steady"]        # duty back to normal, same fleet, nothing else changed
    high = N_REGIMES - 1
    share = {n: float(np.mean(LIVE[n].regime == high)) for n in ("regime", "steady")}
    print(f"readings taken at {REGIME_NAMES[high]} duty: {share['regime']:.0%} of the regime "
          f"window, {share['steady']:.0%} of the steady one.\n")
    print(f"  {'fix on the regime window':<28}{'threshold':>11}{'cost there':>14}"
          f"{'cost when duty returns':>24}")
    for f in ("condition", "reprice"):
        got = row[f]
        feature = adjust_feature(f, later.health, later.regime, later.check, BASELINE)
        counts = alarm_counts(feature, later.runs, got.threshold, LEAD_HOURS)
        print(f"  {f:<28}{got.threshold:>11.2f}{got.cost:>14,.0f}"
              f"{expected_cost(counts, audited.prices):>24,.0f}")
    print("\nsame cost while the plant runs hard, and a different cost the week it stops.")
    print("a threshold raised to absorb a duty change is a threshold that has forgotten")
    print("what it was for. Conditioning carries the reason with it.")


_try("the regime trap", _show_the_regime_trap,
     needs=("exercise 6", "exercise 7", "exercise 8"))

## 12. Closing the loop

The matrix above was priced with numbers the plant measured, not numbers anyone guessed —
and that is the last thing this programme has to say. Module 1 ended with a threshold and
the instruction to re-derive it whenever a price changed. A price *has* changed: two of
them, by the amounts the audit just printed. So re-derive, on the plant's own history,
and measure what the kick-off number is now costing on a fleet where nothing has drifted
at all.

In [ ]:
def _close_the_loop() -> None:
    audited = estimate_prices(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    later = LIVE["steady"]      # a window the drift detector has already called quiet
    rederived = cost_optimal_threshold(REFERENCE.health, REFERENCE.runs, THRESHOLDS,
                                       audited.prices)
    old = alarm_counts(later.health, later.runs, BASELINE.threshold)
    new = alarm_counts(later.health, later.runs, rederived.threshold)
    old_cost = expected_cost(old, audited.prices)
    new_cost = expected_cost(new, audited.prices)
    print(f"  {'':<26}{'threshold':>11}{'caught':>9}{'missed':>9}{'false':>8}{'cost':>14}")
    print(f"  {'kick-off, guessed prices':<26}{BASELINE.threshold:>11.2f}{old.tp:>9}"
          f"{old.fn:>9}{old.fp:>8}{old_cost:>14,.0f}")
    print(f"  {'re-derived, audited prices':<26}{rederived.threshold:>11.2f}{new.tp:>9}"
          f"{new.fn:>9}{new.fp:>8}{new_cost:>14,.0f}")
    print(f"\nboth are scored on the same {len(later.runs)} units of the steady window — "
          f"a fleet neither\nthreshold was fitted on, and one the detector has already "
          f"called quiet. Nothing drifted.\nThe threshold moved by "
          f"{abs(rederived.threshold - BASELINE.threshold):.2f} anyway, because it was never "
          f"a property of the detector:\nit is a property of three prices, and the audit "
          f"found the worst of them out by a factor of\n"
          f"{audited.prices.unplanned / KICKOFF.unplanned:.1f}.")
    if new_cost < old_cost:
        print(f"re-deriving is worth {old_cost - new_cost:,.0f} per {len(later.runs)} units "
              f"per {N_HOURS} h here — {100 * (1 - new_cost / old_cost):.1f}% — and it buys "
              f"{old.fn - new.fn} of the failures back.")
    else:
        print(f"on this fleet the kick-off threshold happens to hold up, by "
              f"{new_cost - old_cost:,.0f}. Re-deriving is still the right policy; what "
              f"changed is that you can now say why.")
    print(f"the price nobody has measured is still "
          f"{', '.join(audited.still_guessed) if audited.still_guessed else 'none of them'}, "
          f"and the note below says so, because a number\nthat came off a whiteboard should "
          f"never travel without a label.")


_try("closing the loop", _close_the_loop, needs=("exercise 7",))

## 13. Common mistakes

- **Choosing the bins from both windows.** Pooled bins move with the drift, and the drift
  disappears. Bins come from the reference and nothing else.
- **Reading a PSI as a probability.** It is not one. It has no distribution you can quote
  without knowing the sample size and the binning, which is why section 3 measured this
  plant's own null band instead of reciting a rule of thumb.
- **Conditioning on something you did not measure when healthy.** A duty point the
  reference never contained has no baseline to condition on. `conditional_psi` raises
  rather than inventing one.
- **Testing duty before the instrument.** A transmitter that reads high makes every other
  statistic a statistic about the transmitter. Section 6's last window shows the two orders
  disagreeing.
- **Treating a drift detector as a model monitor.** Nothing here saw a label. It cannot
  tell you the model got worse; it can tell you the inputs moved, which is all a one-way
  network path allows and more than most plants have.
- **Averaging a price over two observations.** A class the audit barely saw keeps the
  kick-off number *and is reported as a guess*. The alternative is a number with a decimal
  point and no evidence.
- **Re-baselining against the feature instead of against the check.** This is the big one,
  and it is what most condition-monitoring systems do by default: re-estimate each unit's
  quiet level from a rolling window of its own recent readings. Run the next cell.

In [ ]:
def naive_rebaseline(health: np.ndarray, baseline: Baseline,
                     quiet_q: float = QUIET_Q) -> np.ndarray:
    """The rolling re-baseline most monitoring systems ship. Given to you, to price.

    Re-estimate each unit's quiet level from its OWN recent readings — the `quiet_q` quantile
    of this window — and re-express the feature against the kick-off quiet level. It needs no
    check standard, which is why it is popular, and it cannot tell a transmitter that reads
    high from a bearing that is wearing out, which is why the next cell exists.
    """
    h = np.asarray(health, dtype=float)
    return h - (np.quantile(h, float(quiet_q), axis=1) - float(baseline.quiet_level))[:, None]


def _price_the_rolling_baseline() -> None:
    audited = estimate_prices(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    print(f"  {'window':<16}{'fix':<22}{'caught':>8}{'missed':>8}{'false':>7}{'cost':>14}")
    for name in ("recalibration", "ageing"):
        live = LIVE[name]
        checked = adjust_feature("rebaseline", live.health, live.regime, live.check, BASELINE)
        rolled = naive_rebaseline(live.health, BASELINE)
        for label, feature in (("against the check", checked),
                               ("against the feature", rolled)):
            counts = alarm_counts(feature, live.runs, BASELINE.threshold)
            print(f"  {name:<16}{label:<22}{counts.tp:>8}{counts.fn:>8}{counts.fp:>7}"
                  f"{expected_cost(counts, audited.prices):>14,.0f}")
    aged = LIVE["ageing"]
    a = alarm_counts(adjust_feature("rebaseline", aged.health, aged.regime, aged.check,
                                    BASELINE), aged.runs, BASELINE.threshold)
    b = alarm_counts(naive_rebaseline(aged.health, BASELINE), aged.runs, BASELINE.threshold)
    print(f"\non the recalibration window the two are close: the instrument moved, and both")
    print(f"ways of measuring the move find it. On the ageing window the rolling baseline")
    print(f"gives up {b.fn - a.fn} more failures, because every unit's quiet hours are")
    print(f"genuinely higher than they were and it subtracts exactly that.")
    print(f"it is not detecting the fleet's decline. It is normalising it away, one unit at")
    print(f"a time, and the alarm log will look quieter every year.")


_try("the rolling baseline, priced", _price_the_rolling_baseline,
     needs=("exercise 6", "exercise 7"))

## 14. Self-check

1. Your marginal PSI on the pump fleet has tripled. Conditioning on duty leaves it
   unchanged, and the transmitters' self-test readings have not moved. The response is:
   - (a) re-baseline the channels against their current quiet levels
   - (b) retrain the model on the recent window
   - (c) re-derive the threshold, because the base rate has moved and the base rate is an
         input to the cost function

2. A colleague sets the drift alert at a fixed PSI value they found in a blog post. The
   strongest objection is:
   - (a) the value is too low and will alarm constantly
   - (b) a PSI has no meaning independent of the sample size and the binning, so the alert
         level has to be measured on the plant's own reference window
   - (c) PSI should be replaced with a Kolmogorov-Smirnov test

3. Duty and the instrument check both moved after a turnaround. Testing duty first calls
   it a regime change, so you condition on duty. What you have actually done is:
   - (a) compared this week's mis-calibrated readings with a baseline built from correctly
         calibrated ones, inside each regime, and called the difference explained
   - (b) the right thing; conditioning is harmless
   - (c) nothing, because conditioning cannot change a PSI

Before you answer 4 and 5, run this. It prints the three numbers those two questions turn
on, so your answer rests on the notebook's own output rather than on a recollection of it.

In [ ]:
def _self_check_numbers() -> None:
    band = null_psi_band(REFERENCE)
    sig = drift_signature(REFERENCE, LIVE["recal_and_regime"])
    audited = estimate_prices(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    print(f"q2 · this plant's measured alert level: {band:.5f} over {N_BINS} bins on "
          f"{REFERENCE.health.size:,} readings")
    print(f"q3 · the turnaround window: conditional {sig.conditional:.4f}, marginal "
          f"{sig.marginal:.4f}, instrument check {sig.check:.4f}")
    print(f"q4 · audited prices {audited.counts[0]}/{audited.counts[1]}/{audited.counts[2]} "
          f"jobs; still a guess: "
          f"{', '.join(audited.still_guessed) if audited.still_guessed else 'none'}")
    print(f"q5 · the audit is {len(AUDIT)} rows for "
          f"{REFERENCE.health.shape[0] + FLEET['steady'].health.shape[0]} unit-windows of "
          f"history, arriving weeks late")


_try("self-check numbers", _self_check_numbers, needs=_FOR_SIGNATURES + ("exercise 7",))

4. The alarm audit has four false-alarm lines and a minimum of five. Your price estimator
   returns the kick-off guess for that class. Reporting the threshold derived from it
   should therefore say:
   - (a) nothing; it is one price out of three
   - (b) that the estimate is invalid and no threshold can be published
   - (c) that the false-alarm price is still the kick-off number and the threshold is only
         as good as that guess

5. Module 7 established that the plant's data path is one-way, so no labels come back
   automatically. The audit in this lesson does carry outcomes. The reason that is not a
   contradiction is:
   - (a) the audit is a covert return channel through the firewall
   - (b) the audit arrives through people, weeks late and a few dozen rows at a time —
         useless for training a detector, sufficient for pricing a decision
   - (c) the diode only blocks inbound traffic on even-numbered ports

Answers, with the reasoning, are in this lesson's worked solution in the course repository.

One last cell, and it is the deliverable: the note that goes back to the reliability
engineer, carrying the threshold, the prices it came from, which of those prices is still
a guess, and the trigger that will bring you back.

In [ ]:
def _handover() -> None:
    audited = estimate_prices(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    band = null_psi_band(REFERENCE)
    live, history = LIVE["reference"], HISTORY["reference"]
    point = cost_optimal_threshold(history.health, history.runs, THRESHOLDS, audited.prices)
    counts = alarm_counts(live.health, live.runs, point.threshold)
    print("MONITORING PROGRAMME — hand this to the reliability engineer, all six lines")
    print(f"  threshold          health index >= {point.threshold:.2f}, re-derived on "
          f"{len(history.runs)} units of this plant's own history")
    print(f"  lead requirement   {LEAD_HOURS} h of warning, or it is not a catch")
    print(f"  derived from       planned {audited.prices.planned:,.0f} · unplanned "
          f"{audited.prices.unplanned:,.0f} · false alarm {audited.prices.false_alarm:,.0f}")
    still = ", ".join(audited.still_guessed) if audited.still_guessed else "none"
    print(f"  evidence           {audited.counts[0]} / {audited.counts[1]} / "
          f"{audited.counts[2]} audited jobs behind those three prices; still a guess: "
          f"{still}")
    print(f"  drift alert        PSI above {band:.5f} on the feature or the instrument "
          f"self-test, over {N_BINS} reference-quantile bins")
    print(f"  re-derive when     any price is re-measured, the base rate moves, or the "
          f"diagnosis says ageing")
    print(f"  expected outcome   {counts.tp} caught, {counts.fn} missed, {counts.fp} false "
          f"alarms per {len(live.runs)} units per {N_HOURS} h, at "
          f"{expected_cost(counts, audited.prices):,.0f}")


_try("handover note", _handover, needs=("exercise 1", "exercise 2", "exercise 7"))

## What you built, and where it goes next

You now have the loop that module 1 opened: a threshold derived from three prices, three
prices derived from an audit, and an audit fed by the alarms the threshold raised. It turns
over on its own, and each turn replaces one guess with one measurement.

The part worth carrying out of here is the discipline of the diagnosis. Three drifts that
look identical in a summary statistic need three different responses, and the cost of
getting that wrong is not a slightly worse model — this notebook measured it at several
times the cost of the right answer. The question a drift statistic answers is "has
something moved". The question that matters is "which of the three things moved", and you
cannot answer it from the feature alone. You answer it with a variable the process already
records and a check standard the instrument already reports, both of which existed before
anyone mentioned machine learning.

Module 9 is the capstone: a monitoring programme for one asset class, with the feature, the
labelling policy, the alarm rule, the three prices, the deployment constraint and the
re-derivation trigger written down together, because any one of them alone is a number with
no meaning.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<11} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_reference_bins),
                              ("exercise 2", _check_psi),
                              ("exercise 3", _check_conditional_psi),
                              ("exercise 4", _check_drift_signature),
                              ("exercise 5", _check_diagnose_drift),
                              ("exercise 6", _check_adjust_feature),
                              ("exercise 7", _check_estimate_prices),
                              ("exercise 8", _check_apply_fix),
                              ("exercise 9", _check_respond_to_drift)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f} s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))